# LiDAR PointPillars — Google Colab walkthrough

Run this notebook from top to bottom in Google Colab to see the project step by step. It keeps the original Python files unchanged; these cells run and explain them.

> Mock inference works on CPU. Real PointPillars inference needs a GPU runtime, OpenPCDet, and a matching checkpoint/configuration.

## 1. Get the project

Replace `REPOSITORY_URL` if you forked or renamed the repository.

In [ ]:
REPOSITORY_URL = 'https://github.com/harshithanbhat77/lidar-ai-predict.git'
PROJECT_DIRECTORY = 'lidar-ai-predict'

!git clone "$REPOSITORY_URL" "$PROJECT_DIRECTORY"
%cd $PROJECT_DIRECTORY
!find . -maxdepth 2 -type f | sort

## 2. Install dependencies

This installs what is needed for loading files, mock inference, the API, and tests.

In [ ]:
!python -m pip install --upgrade pip
!python -m pip install -r requirements.txt

## 3. Choose a point-cloud file

The repository includes `notebooks/000003.bin` for a quick demo. To use your own KITTI-format file, uncomment the upload lines and set `sample_path` to the uploaded name. A KITTI `.bin` file stores `x, y, z, reflectance` as `float32` values.

In [ ]:
sample_path = 'notebooks/000003.bin'  # included sample

# To upload your own file instead, run these lines and update sample_path:
# from google.colab import files
# uploaded = files.upload()
# sample_path = next(iter(uploaded))

!ls -lh "$sample_path"

## 4. Inspect and visualize the point cloud

This uses the project's `load_kitti_bin` and `summarize_point_cloud` functions, then plots a top-down view.

In [ ]:
import matplotlib.pyplot as plt
from src.pointcloud_loader import load_kitti_bin, summarize_point_cloud

points = load_kitti_bin(sample_path)
summary = summarize_point_cloud(points)
print(summary)
print('Shape:', points.shape, '— columns are x, y, z, reflectance')
display(points[:10])

plot_points = points[::max(1, len(points) // 30000)]
plt.figure(figsize=(10, 8))
plt.scatter(plot_points[:, 0], plot_points[:, 1], c=plot_points[:, 3], s=0.5, cmap='viridis')
plt.colorbar(label='Reflectance')
plt.xlabel('x — forward (m)')
plt.ylabel('y — lateral (m)')
plt.title('Top-down LiDAR point cloud')
plt.axis('equal')
plt.show()

## 5. See the project code

You can inspect any project source file directly in Colab. This shows the file loader and detector wrapper.

In [ ]:
!sed -n '1,220p' src/pointcloud_loader.py
!sed -n '1,260p' src/detector.py

## 6. Run mock inference

Mock mode exercises the complete flow—load file → detector → confidence filtering → JSON response—without claiming to run a trained model.

In [ ]:
from src.detector import DetectorConfig, PointPillarsDetector

detector = PointPillarsDetector(DetectorConfig(mock_mode=True, device='cpu'))
response = detector.predict(points, score_threshold=0.5)
response.model_dump()

In [ ]:
!python -m src.inference --input "$sample_path" --output outputs/mock.json --mock
!cat outputs/mock.json

## 7. Run the automated checks

These tests cover loader validation, post-processing, and the API contract.

In [ ]:
!pytest -q

## 8. Optional: real PointPillars inference

In Colab choose **Runtime → Change runtime type → T4 GPU**. Then install a version of OpenPCDet compatible with Colab's PyTorch/CUDA environment and upload a *matching* PointPillars checkpoint. This project intentionally stops with a clear error until that version-specific OpenPCDet wiring is completed—it never fabricates real detections.

In [ ]:
# Check whether this Colab session has a GPU:
!nvidia-smi

# After installing compatible OpenPCDet and uploading matching files:
# !python -m src.inference --input "$sample_path" --output outputs/real.json \
#     --score-threshold 0.5 --checkpoint models/pointpillars_kitti.pth \
#     --config configs/pointpillars_kitti.yaml